# Outliers Detection - Magic Glasses (MG)

Dedicated notebook for MG outlier detection only:
- `OUTLIER_MAGIC_GLASSES_PARTIAL` (MAD15 -> MAD10)
- `OUTLIER_MAGIC_GLASSES_COMPLETE` (MAD15 -> MAD10 -> seasonal5 -> seasonal3)

In [ ]:
# Parameters with safe defaults
if (!exists("ROOT_PATH")) ROOT_PATH <- "~/workspace"
if (!exists("CONFIG_FILE_NAME")) CONFIG_FILE_NAME <- "SNT_config.json"
# Partial step is always required by MG logic; complete step is optional.
RUN_MAGIC_GLASSES_PARTIAL <- TRUE
if (!exists("RUN_MAGIC_GLASSES_COMPLETE")) RUN_MAGIC_GLASSES_COMPLETE <- FALSE
if (!exists("DEVIATION_MAD15")) DEVIATION_MAD15 <- 15
if (!exists("DEVIATION_MAD10")) DEVIATION_MAD10 <- 10
if (!exists("DEVIATION_SEASONAL5")) DEVIATION_SEASONAL5 <- 5
if (!exists("DEVIATION_SEASONAL3")) DEVIATION_SEASONAL3 <- 3
if (!exists("SEASONAL_WORKERS")) SEASONAL_WORKERS <- 1
if (!exists("DEV_SUBSET")) DEV_SUBSET <- FALSE
if (!exists("DEV_SUBSET_ADM1_N")) DEV_SUBSET_ADM1_N <- 2

if (SEASONAL_WORKERS < 1) {
  stop("SEASONAL_WORKERS must be >= 1")
}

if (DEV_SUBSET_ADM1_N < 1) {
  stop("DEV_SUBSET_ADM1_N must be >= 1")
}

CODE_PATH <- file.path(ROOT_PATH, "code")
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
DATA_PATH <- file.path(ROOT_PATH, "data")
OUTPUT_DIR <- file.path(DATA_PATH, "dhis2", "outliers_imputation")
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
PIPELINE_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_outliers_imputation_magic_glasses")
source(file.path(PIPELINE_PATH, "utils", "snt_dhis2_outliers_imputation_magic_glasses.r"))

required_packages <- c("arrow", "data.table", "jsonlite", "reticulate", "glue")
if (RUN_MAGIC_GLASSES_COMPLETE) {
  required_packages <- c(required_packages, "forecast")
}
if (RUN_MAGIC_GLASSES_COMPLETE && SEASONAL_WORKERS > 1) {
  required_packages <- c(required_packages, "future", "future.apply")
}

setup_ctx <- bootstrap_magic_glasses_context(
  root_path = ROOT_PATH,
  required_packages = required_packages
)

CODE_PATH <- setup_ctx$CODE_PATH
CONFIG_PATH <- setup_ctx$CONFIG_PATH
DATA_PATH <- setup_ctx$DATA_PATH
openhexa <- setup_ctx$openhexa

if (RUN_MAGIC_GLASSES_COMPLETE) {
  log_msg("[WARNING] Complete mode: seasonal detection is very computationally intensive and can take several hours to run.", "warning")
}

if (RUN_MAGIC_GLASSES_COMPLETE && SEASONAL_WORKERS > 1) {
  future::plan(future::multisession, workers = SEASONAL_WORKERS)
  log_msg(glue::glue("Using parallel seasonal detection with {SEASONAL_WORKERS} workers"))
}

config_json <- fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME))
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
fixed_cols <- c("PERIOD", "YEAR", "MONTH", "ADM1_ID", "ADM2_ID", "OU_ID")
indicators_to_keep <- names(config_json$DHIS2_DATA_DEFINITIONS$DHIS2_INDICATOR_DEFINITIONS)

dataset_name <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
dhis2_routine <- get_latest_dataset_file_in_memory(dataset_name, paste0(COUNTRY_CODE, "_routine.parquet"))

cols_to_select <- intersect(c(fixed_cols, indicators_to_keep), names(dhis2_routine))
dt_routine <- as.data.table(dhis2_routine)[, ..cols_to_select]

dhis2_routine_long <- melt(
  dt_routine,
  id.vars = intersect(fixed_cols, names(dt_routine)),
  measure.vars = intersect(indicators_to_keep, names(dt_routine)),
  variable.name = "INDICATOR",
  value.name = "VALUE",
  variable.factor = FALSE
)

if (DEV_SUBSET) {
  unique_adm1 <- unique(dhis2_routine_long$ADM1_ID)
  adm1_to_keep <- unique_adm1[seq_len(min(DEV_SUBSET_ADM1_N, length(unique_adm1)))]
  dhis2_routine_long <- dhis2_routine_long[ADM1_ID %in% adm1_to_keep]
  log_msg(glue::glue("DEV_SUBSET enabled: keeping {length(adm1_to_keep)} ADM1 values"), "warning")
}

log_msg(glue::glue("Data loaded: {nrow(dhis2_routine_long)} rows, {length(unique(dhis2_routine_long$OU_ID))} facilities"))

if (RUN_MAGIC_GLASSES_COMPLETE) {
  n_groups <- uniqueN(dhis2_routine_long[, .(OU_ID, INDICATOR)])
  log_msg(glue::glue("Complete mode active: seasonal detection will run on up to {n_groups} OU_ID x INDICATOR time series."), "warning")
} else {
  log_msg("Partial mode active: seasonal detection is skipped.")
}

In [ ]:
# Helpers loaded from utils/snt_dhis2_outliers_imputation_magic_glasses.r
# - detect_outliers_mad_custom()
# - detect_seasonal_outliers()

In [ ]:
if (RUN_MAGIC_GLASSES_PARTIAL | RUN_MAGIC_GLASSES_COMPLETE) {
  log_msg("Starting MAD15 detection...")
  flagged_outliers_mad15 <- detect_outliers_mad_custom(dhis2_routine_long, DEVIATION_MAD15)
  flagged_outliers_mad15_filtered <- flagged_outliers_mad15[OUTLIER_MAD15 == FALSE]

  log_msg("Starting MAD10 detection...")
  flagged_outliers_mad10 <- detect_outliers_mad_custom(flagged_outliers_mad15_filtered, DEVIATION_MAD10)
  setnames(flagged_outliers_mad10, paste0("OUTLIER_MAD", DEVIATION_MAD10), "OUTLIER_MAD15_MAD10")

  join_cols <- c("PERIOD", "OU_ID", "INDICATOR")
  mad10_subset <- flagged_outliers_mad10[, .(PERIOD, OU_ID, INDICATOR, OUTLIER_MAD15_MAD10)]
  flagged_outliers_mad15_mad10 <- merge(
    flagged_outliers_mad15,
    mad10_subset,
    by = join_cols,
    all.x = TRUE
  )
  flagged_outliers_mad15_mad10[is.na(OUTLIER_MAD15_MAD10), OUTLIER_MAD15_MAD10 := TRUE]
  log_msg(glue::glue("MAD partial done: {sum(flagged_outliers_mad15_mad10$OUTLIER_MAD15_MAD10)} outliers flagged"))
}

if (RUN_MAGIC_GLASSES_COMPLETE) {
  flagged_outliers_mad15_mad10_filtered <- flagged_outliers_mad15_mad10[OUTLIER_MAD15_MAD10 == FALSE]

  if (nrow(flagged_outliers_mad15_mad10_filtered) == 0) {
    log_msg("No rows left after MAD partial filtering; seasonal step will be skipped.", "warning")
    flagged_outliers_seasonal5 <- copy(flagged_outliers_mad15_mad10_filtered)
    flagged_outliers_seasonal5[, OUTLIER_SEASONAL5 := FALSE]
    flagged_outliers_seasonal5_filtered <- flagged_outliers_seasonal5
    flagged_outliers_seasonal3 <- copy(flagged_outliers_seasonal5_filtered)
    flagged_outliers_seasonal3[, OUTLIER_SEASONAL3 := FALSE]
  } else {
    log_msg(glue::glue("Starting SEASONAL5 detection on {nrow(flagged_outliers_mad15_mad10_filtered)} rows..."))
    t_seasonal5 <- system.time({
      flagged_outliers_seasonal5 <- detect_seasonal_outliers(
        flagged_outliers_mad15_mad10_filtered,
        deviation = DEVIATION_SEASONAL5,
        workers = SEASONAL_WORKERS
      )
    })
    flagged_outliers_seasonal5_filtered <- flagged_outliers_seasonal5[OUTLIER_SEASONAL5 == FALSE]
    log_msg(glue::glue("SEASONAL5 finished in {round(t_seasonal5['elapsed'], 1)}s. Remaining rows: {nrow(flagged_outliers_seasonal5_filtered)}"))

    log_msg(glue::glue("Starting SEASONAL3 detection on {nrow(flagged_outliers_seasonal5_filtered)} rows..."))
    t_seasonal3 <- system.time({
      flagged_outliers_seasonal3 <- detect_seasonal_outliers(
        flagged_outliers_seasonal5_filtered,
        deviation = DEVIATION_SEASONAL3,
        workers = SEASONAL_WORKERS
      )
    })
    log_msg(glue::glue("SEASONAL3 finished in {round(t_seasonal3['elapsed'], 1)}s."))
  }

  setnames(flagged_outliers_seasonal3, paste0("OUTLIER_SEASONAL", DEVIATION_SEASONAL3), "OUTLIER_SEASONAL5_SEASONAL3")

  seasonal3_subset <- flagged_outliers_seasonal3[, .(PERIOD, OU_ID, INDICATOR, OUTLIER_SEASONAL5_SEASONAL3)]
  flagged_outliers_seasonal5_seasonal3 <- merge(
    flagged_outliers_seasonal5,
    seasonal3_subset,
    by = join_cols,
    all.x = TRUE
  )
  flagged_outliers_seasonal5_seasonal3[is.na(OUTLIER_SEASONAL5_SEASONAL3), OUTLIER_SEASONAL5_SEASONAL3 := TRUE]
  log_msg(glue::glue("SEASONAL complete done: {sum(flagged_outliers_seasonal5_seasonal3$OUTLIER_SEASONAL5_SEASONAL3)} outliers flagged"))
}

In [ ]:
base_cols <- intersect(c(fixed_cols, "INDICATOR", "VALUE"), names(dhis2_routine_long))
flagged_outliers_mg <- copy(dhis2_routine_long[, ..base_cols])
join_cols <- c("PERIOD", "OU_ID", "INDICATOR")

if (RUN_MAGIC_GLASSES_PARTIAL | RUN_MAGIC_GLASSES_COMPLETE) {
  partial_subset <- flagged_outliers_mad15_mad10[, .(PERIOD, OU_ID, INDICATOR, OUTLIER_MAD15_MAD10)]
  flagged_outliers_mg <- merge(flagged_outliers_mg, partial_subset, by = join_cols, all.x = TRUE)
  setnames(flagged_outliers_mg, "OUTLIER_MAD15_MAD10", "OUTLIER_MAGIC_GLASSES_PARTIAL")
}

if (RUN_MAGIC_GLASSES_COMPLETE) {
  complete_subset <- flagged_outliers_seasonal5_seasonal3[, .(PERIOD, OU_ID, INDICATOR, OUTLIER_SEASONAL5_SEASONAL3)]
  flagged_outliers_mg <- merge(flagged_outliers_mg, complete_subset, by = join_cols, all.x = TRUE)
  setnames(flagged_outliers_mg, "OUTLIER_SEASONAL5_SEASONAL3", "OUTLIER_MAGIC_GLASSES_COMPLETE")
  flagged_outliers_mg[is.na(OUTLIER_MAGIC_GLASSES_COMPLETE) & OUTLIER_MAGIC_GLASSES_PARTIAL == TRUE,
                      OUTLIER_MAGIC_GLASSES_COMPLETE := TRUE]
}

if ("OUTLIER_MAGIC_GLASSES_PARTIAL" %in% names(flagged_outliers_mg)) {
  flagged_outliers_mg[is.na(OUTLIER_MAGIC_GLASSES_PARTIAL), OUTLIER_MAGIC_GLASSES_PARTIAL := FALSE]
}
if ("OUTLIER_MAGIC_GLASSES_COMPLETE" %in% names(flagged_outliers_mg)) {
  flagged_outliers_mg[is.na(OUTLIER_MAGIC_GLASSES_COMPLETE), OUTLIER_MAGIC_GLASSES_COMPLETE := FALSE]
}

active_outlier_col <- if (
  RUN_MAGIC_GLASSES_COMPLETE && "OUTLIER_MAGIC_GLASSES_COMPLETE" %in% names(flagged_outliers_mg)
) {
  "OUTLIER_MAGIC_GLASSES_COMPLETE"
} else {
  "OUTLIER_MAGIC_GLASSES_PARTIAL"
}

if (!(active_outlier_col %in% names(flagged_outliers_mg))) {
  stop(glue::glue("Expected outlier flag column not found: {active_outlier_col}"))
}

pyramid_names <- unique(as.data.table(dhis2_routine)[, .(
  ADM1_NAME, ADM1_ID, ADM2_NAME, ADM2_ID, OU_ID, OU_NAME
)])

# 1) Detected table: full routine data + OUTLIER_DETECTED flag (same structure as mean/median/iqr/path)
outlier_method_label <- if (active_outlier_col == "OUTLIER_MAGIC_GLASSES_COMPLETE") "MAGIC_GLASSES_COMPLETE" else "MAGIC_GLASSES_PARTIAL"
detected_tbl <- flagged_outliers_mg[, .(
  PERIOD, YEAR, MONTH, ADM1_ID, ADM2_ID, OU_ID, INDICATOR, VALUE,
  OUTLIER_DETECTED = get(active_outlier_col),
  OUTLIER_METHOD = outlier_method_label
)]
detected_tbl[is.na(OUTLIER_DETECTED), OUTLIER_DETECTED := FALSE]
detected_tbl <- merge(detected_tbl, unique(pyramid_names), by = c("ADM1_ID", "ADM2_ID", "OU_ID"), all.x = TRUE)
detected_tbl[, DATE := as.Date(sprintf("%04d-%02d-01", YEAR, MONTH))]
arrow::write_parquet(detected_tbl, file.path(OUTPUT_DIR, paste0(COUNTRY_CODE, "_routine_outliers_detected.parquet")))
n_out <- sum(detected_tbl$OUTLIER_DETECTED == TRUE)
log_msg(glue::glue("Exported full detection table ({nrow(detected_tbl)} rows, {n_out} outliers) to {COUNTRY_CODE}_routine_outliers_detected.parquet"))

# 2) Imputed routine data (same moving-average logic as other outlier pipelines)
imputed_long <- copy(flagged_outliers_mg)
setorder(imputed_long, ADM1_ID, ADM2_ID, OU_ID, INDICATOR, PERIOD, YEAR, MONTH)
imputed_long[, TO_IMPUTE := fifelse(get(active_outlier_col) == TRUE, NA_real_, VALUE)]
imputed_long[
  , MOVING_AVG := frollapply(
      TO_IMPUTE,
      n = 3,
      FUN = function(x) ceiling(mean(x, na.rm = TRUE)),
      align = "center"
    ),
  by = .(ADM1_ID, ADM2_ID, OU_ID, INDICATOR)
]
imputed_long[, VALUE_IMPUTED := fifelse(is.na(TO_IMPUTE), MOVING_AVG, TO_IMPUTE)]
imputed_long[, VALUE := VALUE_IMPUTED]
imputed_long[, c("TO_IMPUTE", "MOVING_AVG", "VALUE_IMPUTED") := NULL]

routine_imputed <- to_routine_wide(
  dt_long = imputed_long,
  fixed_cols = fixed_cols,
  indicators_to_keep = indicators_to_keep,
  pyramid_names = pyramid_names
)
arrow::write_parquet(routine_imputed, file.path(OUTPUT_DIR, paste0(COUNTRY_CODE, "_routine_outliers_imputed.parquet")))
log_msg(glue::glue("Exported routine imputed table to {COUNTRY_CODE}_routine_outliers_imputed.parquet"))

# 3) Removed routine data
# We set outlier values to NA (we do not remove rows). The routine data keeps the same structure.
removed_long <- copy(flagged_outliers_mg)
removed_long[get(active_outlier_col) == TRUE, VALUE := NA_real_]

routine_removed <- to_routine_wide(
  dt_long = removed_long,
  fixed_cols = fixed_cols,
  indicators_to_keep = indicators_to_keep,
  pyramid_names = pyramid_names
)
arrow::write_parquet(routine_removed, file.path(OUTPUT_DIR, paste0(COUNTRY_CODE, "_routine_outliers_removed.parquet")))
log_msg(glue::glue("Exported routine removed table to {COUNTRY_CODE}_routine_outliers_removed.parquet"))

log_msg("MG outlier tables exported successfully.")